In [3]:
from pymilvus import Function, FunctionType
from app.core.config import settings
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_community.embeddings import DashScopeEmbeddings
from dotenv import load_dotenv
from langchain_text_splitters import MarkdownHeaderTextSplitter

#===============一、准备工作================
load_dotenv()

def prepare_docs():
    # 读取文档内容
    with open('./resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()
    # 准备切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[ ("#", "h1"), ("##", "h2"), ("###", "h3") ],
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )
    # 切分文档。LangChain写入Milvus的文档：可以定义文档id但必须为字符串类型；可以不定义文档id，则Milvus自动生成，为INT64类型
    return markdown_splitter.split_text(doc)

docs = prepare_docs()
print("> 文档已加载切分完毕")
embedding = DashScopeEmbeddings(model="text-embedding-v4")
print("> 向量模型初始化完毕")

#===============二、初始化向量库================
vectorstore = Milvus(
    embedding,                              # 稠密向量模型
    collection_name="langchain_collection", # collection名称
    builtin_function=BM25BuiltInFunction(   # 生成稀疏向量的函数
        analyzer_params={"type": "chinese"} # 指定中文分词
    ),
    # 【注意】：列表的第一个为稠密向量，第二个为稀疏向量
    vector_field=["dense", "sparse"],       # 向量字段，包括稠密和稀疏，
    connection_args={
        "uri": settings.rag.milvus_url,     # milvus的uri路径
    },
    # index_params=[index_param],           # 可选，索引参数，必须与vector_field对应
    drop_old=True,                          # 是否删除旧的collection，避免重复创建
)
print("> 向量库初始化完毕")

#===============三、文档增删改================
res = vectorstore.add_documents(docs)
print(f"> 添加文档数据成功：{res}")

# res = vectorstore.delete(["468439912009399490"])
# print(f"> 删除文档数据完成：{res}")

#===============四、文档检索================
# 先定义生成ranker的函数，（注意：LangChain 里原始文档字段名是 text，不是官方客户端的 text）
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker",  # ranker名词，唯一即可
        input_field_names=["text"],  # 原始文档字段
        function_type=FunctionType.RERANK,  # ranker类型，这里是固定值
        params={
            "reranker": "model",  # rerank类型，基于模型rerank，也就是cross-encoder
            "model_name": "gte-rerank-v2",  # rerank模型名称
            "provider": "ali",  # rerank模型提供者
            "queries": queries,  # 查询条件
            "max_client_batch_size": 5,  # 向模型发送请求时的批处理限制
        },
    )

# 执行混合检索
query = "苏霍姆林斯基著有哪本书？"
results = vectorstore.similarity_search_with_score(
        query=query,
        k=3,            # 最终返回的TopK文档数
        fetch_k=5,      # 初筛时召回的文档的数量,每路最多fetch_k个
        # expr='h2 like "%教育的功能%"',
        reranker=create_cross_encoder_ranker([query])
    )
for doc, score in results:
    print(f"============score: {score}==============")
    print(doc.model_dump_json(indent=2))

> 文档已加载切分完毕
> 向量模型初始化完毕
> 向量库初始化完毕
> 添加文档数据成功：[470200415637511320, 470200415637511321, 470200415637511322, 470200415637511323, 470200415637511324, 470200415637511325, 470200415637511326, 470200415637511327, 470200415637511328, 470200415637511329, 470200415637511330, 470200415637511331, 470200415637511332, 470200415637511333, 470200415637511334, 470200415637511335, 470200415637511336, 470200415637511337, 470200415637511338, 470200415637511339, 470200415637511340]
============score: 0.18669073283672333==============
{
  "id": null,
  "metadata": {
    "h2": "第一节 中外教育家及其教育思想",
    "h3": "（八）教育学分化时期代表人物及主要思想",
    "h1": "第一章 教育概述",
    "pk": 470200415637511327
  },
  "page_content": "### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，